Get data

In [2]:
#import package
import pandas as pd
from src.dataset import loader, data_info
from src.eda import data_profiling
from src.dictionaries import dictionary
from src.pre_processing import fillna_value

In [3]:
#Get data
article_id = "23690811"
api_url = f"https://api.figshare.com/v2/articles/{article_id}"
dataset = loader.get_data(api_url)
data_info.dataset_info(dataset)

TravelDataUsers = dataset["B_TravelDataUsers.csv"]
TravelDataJourneys = dataset["A_TravelDataJourneys.csv"]

dict_keys(['A_TravelDataJourneys.csv', 'B_TravelDataUsers.csv', 'C_Codebook.xlsx', 'D_PreppedLogFile.xlsx'])
Tổng file: 4
Tên file: A_TravelDataJourneys.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2456414 entries, 0 to 2456413
Data columns (total 10 columns):
 #   Column        Dtype  
---  ------        -----  
 0   UserID        int64  
 1   PurchaseID    int64  
 2   TIMESPSS      object 
 3   Duration      float64
 4   DEVICE_TYPE   object 
 5   type_touch    int64  
 6   purchase_own  int64  
 7   purchase_any  int64  
 8   MobilePanel   int64  
 9   FixedPanel    int64  
dtypes: float64(1), int64(7), object(2)
memory usage: 187.4+ MB
file info: None
Dữ liệu mẫu
   UserID  PurchaseID             TIMESPSS  Duration DEVICE_TYPE  type_touch  \
0    9558           1  2016-03-12 22:18:34     5.786      MOBILE           7   
1    9558           1  2016-03-13 05:25:49    65.725      MOBILE           7   
2    9558           1  2016-03-13 12:26:21     6.492      MOBILE          

In [4]:
trigger_profiling = input('Do you want to get profiling data?(y/n)')

if trigger_profiling =='y':
    # User Dashboard information
    output_dir = "../resources/profiling"
    user_output_filename = "TravelDataUsers_data_profiling.html"
    user_dashboard_title = "Travel Users Dataset Profiling"

    #Run
    data_profiling(TravelDataUsers, output_dir,user_output_filename, user_dashboard_title)

    #Journey Dashboard information
    output_dir = "../resources/profiling"
    journey_output_filename = "TravelDataJourneys_data_profiling.html"
    journey_dashboard_title = "Travel Journeys Dataset Profiling"

    #Run
    data_profiling(TravelDataJourneys, output_dir,journey_output_filename, journey_dashboard_title)
else:
    print('Nothing to do!')

Nothing to do!


In [6]:
#Preparing data for Clustering
user_data = TravelDataUsers[~TravelDataUsers['SPSS_Regio5'].isnull()]
selected_column = [
    'UserID',
    'GenderID', #gioi tinh
    'Age', #tuoi
    'SPSS_Regio5', #vung
    'BAS_huishoudgrootte', #quy mo ho gia dinh (so nguoi trong nha)
    'BAS_werkzaamheid_resp', #nghe nghiep
    'BAS_bruto_jaarinkomen', #tong thu nhap truoc thue
    'afg_kinderen_huishouden', #so luong tre em trong gia dinh
    'AFG_sk2015', #tang lop xa hoi
    'BAS_voltooide_opleiding8_resp', #trinh do hoc van
    'SPSS_Lifestage' #giai doan cuoc doi
]

user_data = user_data[selected_column]
user_data = fillna_value(user_data, ["BAS_voltooide_opleiding8_resp"], 97)
user_data.info()

BAS_voltooide_opleiding8_resp    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 8075 entries, 2 to 9677
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   UserID                         8075 non-null   int64  
 1   GenderID                       8075 non-null   float64
 2   Age                            8075 non-null   float64
 3   SPSS_Regio5                    8075 non-null   float64
 4   BAS_huishoudgrootte            8075 non-null   float64
 5   BAS_werkzaamheid_resp          8075 non-null   float64
 6   BAS_bruto_jaarinkomen          8075 non-null   float64
 7   afg_kinderen_huishouden        8075 non-null   float64
 8   AFG_sk2015                     8075 non-null   float64
 9   BAS_voltooide_opleiding8_resp  8075 non-null   float64
 10  SPSS_Lifestage                 8075 non-null   float64
dtypes: float64(10), int64(1)
memory usage: 757.0 KB


In [23]:
#Preparing data for Prediction
#Dictionary mapping
journey_data = TravelDataJourneys[TravelDataJourneys['Duration'] > 0].copy()
#Duration > 0
journey_data['TIMESPSS'] = pd.to_datetime(journey_data['TIMESPSS'])
journey_data = journey_data.sort_values(by=['UserID', 'TIMESPSS']).reset_index(drop=True)

#Đánh dấu các click liên tiếp cùng loại của cùng 1 User
change_mask = (journey_data['type_touch'] != journey_data['type_touch'].shift()) | (journey_data['UserID'] != journey_data['UserID'].shift())
journey_data['group_id'] = change_mask.cumsum()

#Aggregation
journey_data_compressed = journey_data.groupby(['UserID', 'group_id', 'type_touch']).agg({
    'TIMESPSS': 'first',
    'Duration': 'sum',
    'PurchaseID': 'first',
    'DEVICE_TYPE': 'first',
    'purchase_own': 'max',
    'purchase_any': 'max',
    'MobilePanel': 'first',
    'FixedPanel': 'first'
}).reset_index()

# 5. Mapping tên điểm chạm và Join thông tin User
# Mapping bằng dictionary
user_id_list = user_data['UserID'].unique()
journey_data_compressed = journey_data_compressed[journey_data_compressed['UserID'].isin(user_id_list)]
dictionary = dictionary()
touchpoint_mapping = dictionary.type_touch
journey_data_compressed['touchpoint_name'] = journey_data_compressed['type_touch'].map(touchpoint_mapping)
# # 6. Dọn dẹp và kiểm tra
journey_data_compressed = journey_data_compressed.drop(columns='group_id')

print(f"Số dòng sau khi nén: {len(journey_data_compressed)}")

Số dòng sau khi nén: 268448


In [8]:
journey_data_compressed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 268448 entries, 0 to 303085
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   UserID        268448 non-null  int64         
 1   type_touch    268448 non-null  int64         
 2   TIMESPSS      268448 non-null  datetime64[ns]
 3   Duration      268448 non-null  float64       
 4   PurchaseID    268448 non-null  int64         
 5   DEVICE_TYPE   268448 non-null  object        
 6   purchase_own  268448 non-null  int64         
 7   purchase_any  268448 non-null  int64         
 8   MobilePanel   268448 non-null  int64         
 9   FixedPanel    268448 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(7), object(1)
memory usage: 22.5+ MB
